# Scoring threshold tuning + hold-out gate

Calibrate `ideal` / `limit` bands for `config/scoring.yaml` against batch metrics
and expert labels. **Heuristics remain the scorer** — this notebook is a
calibration search, not a learned model (`docs/ML_UPGRADE.md`, first slice items 2–3).

**This notebook never writes `config/scoring.yaml`.** Even if the hold-out gate
prints `SAFE TO PROMOTE`, copy the YAML snippet by hand. Promote only when
hold-out agreement rises.

### Workflow
1. Drop labeled images under `data/raw/` (`excellent/`, `good/`, `warning/`, `critical/`).
2. From the repo root, run the batch CLI:
   ```bash
   python -m technique_titan.batch.process_folder \
     --input data/raw --output data/processed --labels data/labels.csv
   ```
3. Load the **frozen** hold-out split (`data/eval/holdout_split.json`). Never reshuffle
   if that file exists (`load_or_create_split`).
4. Measure **current** YAML agreement with `evaluate()` (train vs **holdout** vs all).
5. Fit candidate `ideal` / `limit` bands on **train filenames only**.
6. Re-score every row with `score_all` / `score_metric` (weights unchanged).
7. **Hold-out gate:** `SAFE TO PROMOTE` only if hold-out macro accuracy rises and
   macro κ does not drop by more than 0.02; otherwise `DO NOT PROMOTE`.

**Setup** (once, from repo root, Python 3.11 venv):
```bash
pip install -e .
pip install -r requirements-dev.txt
```

If `data/processed/batch_summary.csv` is missing, run the batch command above.
Scoring / agreement cells skip until that file exists.



In [ ]:
from __future__ import annotations

import copy
import sys
import tempfile
from pathlib import Path

try:
    from IPython.display import display
except ImportError:  # plain Python fallback
    def display(obj):
        print(obj)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

_ROOT_GUESS = Path.cwd()
if not (_ROOT_GUESS / "config" / "scoring.yaml").exists():
    _ROOT_GUESS = _ROOT_GUESS.parent
_SRC = _ROOT_GUESS / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from technique_titan.scoring import load_config, score_all, score_metric, severity

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

try:
    from technique_titan.eval import (
        CRITERIA,
        SEVERITIES,
        merge_predictions_and_labels,
        evaluate,
        load_or_create_split,
        cohen_kappa,
        accuracy,
        confusion_matrix,
    )

    HAS_EVAL = True
except ImportError as exc:
    HAS_EVAL = False
    CRITERIA = (
        "wrist_height",
        "finger_curvature",
        "thumb_position",
        "wrist_lateral",
        "hand_arch",
    )
    SEVERITIES = ("good", "warning", "critical")
    print(
        "technique_titan.eval is not importable yet "
        f"({exc}). Split / agreement / gate cells will skip."
    )

ROOT = _ROOT_GUESS

CONFIG_PATH = ROOT / "config" / "scoring.yaml"
SUMMARY_PATH = ROOT / "data" / "processed" / "batch_summary.csv"
LABELS_PATH = ROOT / "data" / "labels.csv"
SPLIT_PATH = ROOT / "data" / "eval" / "holdout_split.json"
REPORTS_DIR = ROOT / "data" / "eval" / "reports"
BATCH_CMD = (
    "python -m technique_titan.batch.process_folder "
    "--input data/raw --output data/processed --labels data/labels.csv"
)

config = load_config(CONFIG_PATH)
criteria = config["criteria"]
bands = config["severity_bands"]
CANDIDATE = copy.deepcopy(config)

HAS_LABELS = LABELS_PATH.exists()
HAS_SUMMARY = SUMMARY_PATH.exists()
df: pd.DataFrame | None = None
split: dict | None = None
baseline_report: dict | None = None
candidate_report: dict | None = None
matched_df: pd.DataFrame | None = None
scored: pd.DataFrame | None = None
PROMOTE_VERDICT = "DO NOT PROMOTE"
GATE_REASON = "gate not run yet"
FIT_DETAILS = pd.DataFrame()

print(f"repo root : {ROOT}")
print(f"config    : {CONFIG_PATH}")
print(f"labels    : {LABELS_PATH} ({'found' if HAS_LABELS else 'missing'})")
print(f"split     : {SPLIT_PATH} ({'found' if SPLIT_PATH.exists() else 'will be created if eval is available'})")
print(
    f"summary   : {SUMMARY_PATH} "
    f"({'found' if HAS_SUMMARY else 'missing — run batch CLI first'})"
)
if not HAS_SUMMARY:
    print("Run from repo root:")
    print(f"  {BATCH_CMD}")



## 1. Current thresholds

Each criterion scores **100** inside `ideal`, **0** at/beyond `limit`, and linearly
in between. Severity uses `severity_bands` (`good` ≥ `good_min`, `warning` ≥
`warning_min`, else `critical`). Weights are left unchanged in this search.



In [ ]:
threshold_rows = []
for name, cfg in criteria.items():
    threshold_rows.append(
        {
            "criterion": name,
            "metric": cfg["metric"],
            "ideal_lo": cfg["ideal"][0],
            "ideal_hi": cfg["ideal"][1],
            "limit_lo": cfg["limit"][0],
            "limit_hi": cfg["limit"][1],
            "weight": cfg["weight"],
        }
    )
thresholds = pd.DataFrame(threshold_rows).set_index("criterion")
thresholds



## 2. Load batch summary

Expects `data/processed/batch_summary.csv` from the batch CLI (one row per
detected hand). If it is missing, later scoring cells skip instead of crashing.



In [ ]:
if not HAS_SUMMARY:
    print("No batch summary — skipping load.")
    print("Add images under data/raw/ then run:")
    print(f"  {BATCH_CMD}")
    df = None
else:
    df = pd.read_csv(SUMMARY_PATH)
    print(f"{len(df)} hand row(s), {df['source'].nunique()} image(s)")
    display(df.head())



## 3. Metric distributions vs ideal / limit bands

Green = current ideal (score 100). Orange dashed = current limit edges (score 0).



In [ ]:
def plot_metric_bands(
    frame: pd.DataFrame,
    scoring_cfg: dict,
    candidate_cfg: dict | None = None,
) -> pd.DataFrame:
    items = list(scoring_cfg["criteria"].items())
    fig, axes = plt.subplots(len(items), 1, figsize=(9, 2.4 * len(items)), sharex=False)
    if len(items) == 1:
        axes = [axes]

    for ax, (name, cfg) in zip(axes, items):
        metric = cfg["metric"]
        if metric not in frame.columns:
            ax.set_title(f"{name}: missing column {metric}")
            continue
        values = frame[metric].dropna()
        ax.hist(
            values,
            bins=min(30, max(8, len(values) // 2)),
            color="#4a6fa5",
            alpha=0.85,
            edgecolor="white",
        )
        ideal_lo, ideal_hi = cfg["ideal"]
        limit_lo, limit_hi = cfg["limit"]
        ax.axvspan(ideal_lo, ideal_hi, color="#4caf50", alpha=0.18, label="ideal")
        ax.axvline(limit_lo, color="#e67e22", ls="--", lw=1.5, label="limit")
        ax.axvline(limit_hi, color="#e67e22", ls="--", lw=1.5)
        if candidate_cfg is not None:
            cc = candidate_cfg["criteria"][name]
            ax.axvspan(cc["ideal"][0], cc["ideal"][1], color="#c0392b", alpha=0.12, label="candidate ideal")
            ax.axvline(cc["limit"][0], color="#c0392b", ls=":", lw=1.2, label="candidate limit")
            ax.axvline(cc["limit"][1], color="#c0392b", ls=":", lw=1.2)
        ax.set_ylabel("count")
        ax.set_title(f"{name}  ({metric})  n={len(values)}")
        ax.legend(loc="upper right", fontsize=8)

    axes[-1].set_xlabel("metric value")
    fig.tight_layout()
    plt.show()
    metric_cols = [c["metric"] for c in scoring_cfg["criteria"].values() if c["metric"] in frame.columns]
    return frame[metric_cols].describe().T


if df is None:
    print("Skipping distributions: no batch summary.")
    print(f"  {BATCH_CMD}")
else:
    display(plot_metric_bands(df, config))



## 4. Frozen hold-out split

Split is by **filename** (not hand-row), stored at `data/eval/holdout_split.json`.
`load_or_create_split` loads that file when it exists and **does not reshuffle**.



In [ ]:
if not HAS_EVAL:
    print("Skipping split: technique_titan.eval is not importable.")
elif not HAS_LABELS:
    print(f"Skipping split: labels not found at {LABELS_PATH}")
else:
    SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)
    split = load_or_create_split(LABELS_PATH, SPLIT_PATH)
    print(f"split file : {SPLIT_PATH}")
    print(f"seed={split.get('seed')}  holdout_fraction={split.get('holdout_fraction')}")
    print(f"train   : {len(split['train'])} file(s)")
    print(f"holdout : {len(split['holdout'])} file(s)")
    print("train:", ", ".join(split["train"][:8]) + (" …" if len(split["train"]) > 8 else ""))
    print("holdout:", ", ".join(split["holdout"][:8]) + (" …" if len(split["holdout"]) > 8 else ""))



## 5. Current heuristic agreement (baseline)

`evaluate()` merges `batch_summary.csv` with `labels.csv` on filename + hand and
reports per-criterion accuracy and Cohen’s κ on **train**, **holdout**, and **all**.
This is the baseline the candidate must beat on hold-out.

NFR-ACC-2 is ≥85% hold-out agreement. This cell reports the number; it does not
claim the NFR is met unless hold-out macro accuracy is actually ≥ 0.85.



In [ ]:
def agreement_frame(report: dict) -> pd.DataFrame:
    rows = []
    for name in CRITERIA:
        block = report.get("criteria", {}).get(name, {})
        for sl in ("train", "holdout", "all"):
            cell = block.get(sl) or {}
            rows.append(
                {
                    "criterion": name,
                    "slice": sl,
                    "n": cell.get("n"),
                    "accuracy": cell.get("accuracy"),
                    "kappa": cell.get("kappa"),
                }
            )
    return pd.DataFrame(rows)


def macro_frame(report: dict) -> pd.DataFrame:
    rows = []
    for sl in ("train", "holdout", "all"):
        cell = (report.get("macro") or {}).get(sl) or {}
        rows.append(
            {
                "slice": sl,
                "accuracy": cell.get("accuracy"),
                "kappa": cell.get("kappa"),
            }
        )
    return pd.DataFrame(rows)


def nfr_acc2_line(report: dict, label: str) -> str:
    hold = (report.get("macro") or {}).get("holdout") or {}
    acc = hold.get("accuracy")
    if acc is None or (isinstance(acc, float) and not np.isfinite(acc)):
        return f"NFR-ACC-2 (≥85% hold-out): not evaluated for {label}"
    met = acc >= 0.85
    status = "MET" if met else "NOT MET"
    return f"NFR-ACC-2 (≥85% hold-out): {status} for {label} (macro acc={acc:.3f})"


def _macro(report: dict, sl: str) -> tuple[float, float]:
    cell = (report.get("macro") or {}).get(sl) or {}
    return float(cell.get("accuracy", float("nan"))), float(cell.get("kappa", float("nan")))


def show_confusion(report: dict, sl: str = "holdout") -> None:
    print(f"{sl} confusion (rows=true, cols=pred; {', '.join(SEVERITIES)})")
    for name in CRITERIA:
        cm = ((report.get("criteria") or {}).get(name, {}).get(sl) or {}).get("confusion") or {}
        labels = cm.get("labels") or list(SEVERITIES)
        matrix = cm.get("matrix")
        if not matrix:
            continue
        print(name)
        display(pd.DataFrame(matrix, index=labels, columns=labels))


if not HAS_EVAL:
    print("Skipping baseline: technique_titan.eval is not importable.")
elif not HAS_SUMMARY:
    print("Skipping baseline: no batch summary.")
    print(f"  {BATCH_CMD}")
elif not HAS_LABELS:
    print(f"Skipping baseline: labels not found at {LABELS_PATH}")
elif split is None:
    print("Skipping baseline: run the split cell first.")
else:
    baseline_report = evaluate(SUMMARY_PATH, LABELS_PATH, SPLIT_PATH, output_dir=None)
    print(
        f"matched={baseline_report.get('n_matched')}  "
        f"unmatched_pred={baseline_report.get('n_unmatched_predictions')}  "
        f"unmatched_labels={baseline_report.get('n_unmatched_labels')}"
    )
    print("Per-criterion accuracy + κ")
    display(agreement_frame(baseline_report).round(3))
    print("Macro (unweighted mean of criteria with n>0)")
    display(macro_frame(baseline_report).round(3))
    print(nfr_acc2_line(baseline_report, "current YAML"))
    show_confusion(baseline_report, "holdout")



## 6. Fit candidate `ideal` / `limit` bands (train only)

For each criterion, take labeled **train** rows and the raw metric column named in
`scoring.yaml` (`criteria.*.metric`, e.g. `wrist_height_delta`). Group by expert
severity.

| Band | How it is proposed |
|---|---|
| `ideal` | Central mass of expert-`good` values (percentile pairs: 5–95, 10–90, 15–85, 20–80, 25–75, 30–70) |
| `limit` | Span of train metrics and of warning/critical, plus small expansions |

Always enforce `limit_lo < ideal_lo <= ideal_hi < limit_hi`. Current YAML bands
are in the grid. Each candidate is **re-scored on train only** with `score_metric`
+ existing `severity_bands`; `accuracy` is the primary pick, `cohen_kappa` the
tie-break, then smaller L1 change from current YAML.

**Hold-out rows are never used to choose bands.** Weights are not changed.



In [ ]:
def rescore_frame(frame: pd.DataFrame, scoring_cfg: dict, prefix: str) -> pd.DataFrame:
    """Apply scoring.yaml logic to each row; returns score/severity/composite columns."""
    out = frame.copy()
    score_cols: dict[str, list] = {}
    sev_cols: dict[str, list] = {}
    composites = []

    for _, row in frame.iterrows():
        metrics = {cfg["metric"]: row.get(cfg["metric"]) for cfg in scoring_cfg["criteria"].values()}
        scored = score_all(metrics, scoring_cfg)
        for criterion, value in scored["scores"].items():
            score_cols.setdefault(criterion, []).append(value)
        for criterion, value in scored["severities"].items():
            sev_cols.setdefault(criterion, []).append(value)
        composites.append(scored["composite_score"])

    for criterion, values in score_cols.items():
        out[f"{prefix}_score_{criterion}"] = values
    for criterion, values in sev_cols.items():
        out[f"{prefix}_severity_{criterion}"] = values
    out[f"{prefix}_composite"] = composites
    return out


def _eps_for_metric(values: np.ndarray) -> float:
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if len(finite) == 0:
        return 1e-3
    span = float(np.max(finite) - np.min(finite))
    if span <= 0:
        span = abs(float(np.median(finite))) or 1.0
    return max(span * 0.01, 1e-6)


def _round_threshold(value: float) -> float:
    av = abs(value)
    if av >= 50:
        nd = 1
    elif av >= 5:
        nd = 2
    elif av >= 0.5:
        nd = 3
    else:
        nd = 4
    return round(float(value), nd)


def _ordered_bands(ideal_lo, ideal_hi, limit_lo, limit_hi, eps):
    ideal_lo, ideal_hi = float(ideal_lo), float(ideal_hi)
    limit_lo, limit_hi = float(limit_lo), float(limit_hi)
    if ideal_lo > ideal_hi:
        ideal_lo, ideal_hi = ideal_hi, ideal_lo
    if ideal_hi - ideal_lo < eps:
        mid = 0.5 * (ideal_lo + ideal_hi)
        ideal_lo, ideal_hi = mid - eps, mid + eps
    gap = max(eps, 0.5 * (ideal_hi - ideal_lo))
    if not (limit_lo < ideal_lo):
        limit_lo = ideal_lo - gap
    if not (limit_hi > ideal_hi):
        limit_hi = ideal_hi + gap
    return ideal_lo, ideal_hi, limit_lo, limit_hi


def _percentile(values: np.ndarray, q: float) -> float:
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    return float(np.percentile(finite, q))


def _band_grid(values: np.ndarray, y_true: np.ndarray, current_ideal, current_limit):
    values = np.asarray(values, dtype=float)
    y_true = np.asarray(y_true)
    eps = _eps_for_metric(values)
    good = values[(y_true == "good") & np.isfinite(values)]
    warn_crit = values[((y_true == "warning") | (y_true == "critical")) & np.isfinite(values)]
    all_v = values[np.isfinite(values)]

    ideals: list[tuple[float, float]] = [(float(current_ideal[0]), float(current_ideal[1]))]
    if len(good) >= 2:
        for plo, phi in ((5, 95), (10, 90), (15, 85), (20, 80), (25, 75), (30, 70)):
            ideals.append((_percentile(good, plo), _percentile(good, phi)))
    elif len(good) == 1:
        g = float(good[0])
        ideals.append((g - eps, g + eps))

    limits: list[tuple[float, float]] = [(float(current_limit[0]), float(current_limit[1]))]
    if len(all_v):
        limits.append((float(np.min(all_v)), float(np.max(all_v))))
        if len(all_v) >= 5:
            limits.append((_percentile(all_v, 1), _percentile(all_v, 99)))
            limits.append((_percentile(all_v, 5), _percentile(all_v, 95)))
    if len(warn_crit):
        limits.append((float(np.min(warn_crit)), float(np.max(warn_crit))))

    span_all = float(np.max(all_v) - np.min(all_v)) if len(all_v) else eps
    raw: list[tuple[float, float, float, float]] = []
    raw.append(_ordered_bands(current_ideal[0], current_ideal[1], current_limit[0], current_limit[1], eps))
    for ilo, ihi in ideals:
        for llo, lhi in limits:
            for frac in (0.0, 0.1, 0.25, 0.5):
                margin = frac * (span_all if span_all > 0 else eps)
                raw.append(_ordered_bands(ilo, ihi, llo - margin, lhi + margin, eps))

    uniq: list[tuple[float, float, float, float]] = []
    seen: set[tuple] = set()
    for band in raw:
        rounded = tuple(_round_threshold(x) for x in band)
        rounded = tuple(_round_threshold(x) for x in _ordered_bands(*rounded, eps))
        if rounded not in seen:
            seen.add(rounded)
            uniq.append(rounded)  # type: ignore[arg-type]
    return uniq


def _train_agreement(values, y_true, ideal, limit, sev_bands):
    y_keep: list[str] = []
    y_pred: list[str] = []
    for v, t in zip(values, y_true):
        if t not in SEVERITIES or not np.isfinite(v):
            continue
        pred = severity(score_metric(float(v), list(ideal), list(limit)), sev_bands)
        if pred not in SEVERITIES:
            continue
        y_keep.append(t)
        y_pred.append(pred)
    n = len(y_keep)
    if n == 0:
        return -1.0, -1.0, 0
    return float(accuracy(y_keep, y_pred)), float(cohen_kappa(y_keep, y_pred, labels=SEVERITIES)), n


def attach_metrics(matched: list[dict], summary: pd.DataFrame) -> pd.DataFrame:
    merged = pd.DataFrame(matched)
    metric_cols = [criteria[c]["metric"] for c in CRITERIA if criteria[c]["metric"] in summary.columns]
    extra = summary[["source", "hand", *metric_cols]].rename(columns={"source": "filename"})
    keep_metrics = [c for c in metric_cols if c not in merged.columns]
    extra = extra[["filename", "hand", *keep_metrics]].drop_duplicates(subset=["filename", "hand"])
    if keep_metrics:
        return merged.merge(extra, on=["filename", "hand"], how="left")
    return merged


if not HAS_EVAL:
    print("Skipping fit: technique_titan.eval is not importable.")
elif df is None:
    print("Skipping fit: no batch summary.")
    print(f"  {BATCH_CMD}")
elif not HAS_LABELS:
    print("Skipping fit: labels.csv missing.")
elif split is None:
    print("Skipping fit: run the split cell first.")
else:
    labels_rows = pd.read_csv(LABELS_PATH).to_dict(orient="records")
    summary_rows = df.to_dict(orient="records")
    matched = merge_predictions_and_labels(summary_rows, labels_rows, match_hand=True)
    matched_df = attach_metrics(matched, df)
    train_names = set(split["train"])
    hold_names = set(split["holdout"])
    matched_df["split"] = matched_df["filename"].map(
        lambda fn: "holdout" if fn in hold_names else ("train" if fn in train_names else "other")
    )
    train_df = matched_df[matched_df["split"] == "train"].copy()
    print(
        f"matched rows={len(matched_df)}  "
        f"train={int((matched_df['split']=='train').sum())}  "
        f"holdout={int((matched_df['split']=='holdout').sum())}"
    )

    CANDIDATE = copy.deepcopy(config)
    fit_rows = []
    for name in CRITERIA:
        cfg = criteria[name]
        metric = cfg["metric"]
        true_col = f"true_{name}"
        if metric not in train_df.columns or true_col not in train_df.columns:
            print(f"{name}: missing {metric!r} or {true_col!r} — keeping current bands")
            continue
        sub = train_df.dropna(subset=[metric, true_col])
        sub = sub[sub[true_col].isin(SEVERITIES)]
        if sub.empty:
            print(f"{name}: no labeled train rows — keeping current bands")
            continue
        values = sub[metric].to_numpy(dtype=float)
        y_true = sub[true_col].to_numpy()
        current = (
            float(cfg["ideal"][0]),
            float(cfg["ideal"][1]),
            float(cfg["limit"][0]),
            float(cfg["limit"][1]),
        )
        best = None
        for ideal_lo, ideal_hi, limit_lo, limit_hi in _band_grid(values, y_true, cfg["ideal"], cfg["limit"]):
            acc, kap, n = _train_agreement(
                values, y_true, [ideal_lo, ideal_hi], [limit_lo, limit_hi], bands
            )
            l1 = sum(
                abs(a - b)
                for a, b in zip((ideal_lo, ideal_hi, limit_lo, limit_hi), current)
            )
            score = (acc, kap, -l1)
            cand = {
                "criterion": name,
                "metric": metric,
                "n_train": n,
                "train_acc": acc,
                "train_kappa": kap,
                "ideal_lo": ideal_lo,
                "ideal_hi": ideal_hi,
                "limit_lo": limit_lo,
                "limit_hi": limit_hi,
                "l1_from_current": l1,
            }
            if best is None or score > best[0]:
                best = (score, cand)
        assert best is not None
        winner = best[1]
        CANDIDATE["criteria"][name]["ideal"] = [winner["ideal_lo"], winner["ideal_hi"]]
        CANDIDATE["criteria"][name]["limit"] = [winner["limit_lo"], winner["limit_hi"]]
        cur_acc, cur_kap, _ = _train_agreement(
            values, y_true, cfg["ideal"], cfg["limit"], bands
        )
        winner["current_train_acc"] = cur_acc
        winner["current_train_kappa"] = cur_kap
        fit_rows.append(winner)

    FIT_DETAILS = pd.DataFrame(fit_rows)
    print("Train-only band search (winner per criterion)")
    display(FIT_DETAILS.round(4))
    print("Candidate vs current YAML")
    compare_bands = thresholds.copy()
    for name in CRITERIA:
        cc = CANDIDATE["criteria"][name]
        compare_bands.loc[name, "cand_ideal_lo"] = cc["ideal"][0]
        compare_bands.loc[name, "cand_ideal_hi"] = cc["ideal"][1]
        compare_bands.loc[name, "cand_limit_lo"] = cc["limit"][0]
        compare_bands.loc[name, "cand_limit_hi"] = cc["limit"][1]
    display(compare_bands)
    if df is not None:
        print("Distributions with candidate overlay (red dotted = candidate limit)")
        display(plot_metric_bands(df, config, CANDIDATE))



## 7. What-if: re-score with candidate bands

`CANDIDATE` starts as the train-fitted bands. Uncomment overrides below if you
want to edit a range by hand, then re-run this cell and the gate.

Re-scoring uses `score_all` / `score_metric` and the YAML `severity_bands`.
Weights are unchanged.



In [ ]:
# Optional manual overrides (re-run this cell + the hold-out gate after editing):
# CANDIDATE["criteria"]["wrist_height"]["ideal"] = [-0.08, 0.18]
# CANDIDATE["criteria"]["wrist_height"]["limit"] = [-0.45, 0.55]
# CANDIDATE["criteria"]["finger_curvature"]["ideal"] = [110.0, 155.0]
# Do not change weights unless you have a clear reason.
# CANDIDATE["severity_bands"]["good_min"] = 75

if df is None:
    print("Skipping re-score: no batch summary.")
    print(f"  {BATCH_CMD}")
    scored = None
else:
    scored = rescore_frame(df, config, "current")
    scored = rescore_frame(scored, CANDIDATE, "candidate")
    compare = pd.DataFrame(
        {
            "current_composite_mean": [scored["current_composite"].mean()],
            "candidate_composite_mean": [scored["candidate_composite"].mean()],
            "current_composite_median": [scored["current_composite"].median()],
            "candidate_composite_median": [scored["candidate_composite"].median()],
        }
    )
    display(compare)



In [ ]:
def severity_mix(frame: pd.DataFrame, prefix: str) -> pd.DataFrame:
    cols = [c for c in frame.columns if c.startswith(f"{prefix}_severity_")]
    mixes = {}
    for col in cols:
        criterion = col.replace(f"{prefix}_severity_", "")
        mixes[criterion] = frame[col].value_counts(normalize=True).mul(100).round(1)
    return pd.DataFrame(mixes).fillna(0.0)


if scored is None:
    print("Skipping severity mix: no re-score.")
else:
    print("Current severity mix (% of hands)")
    display(severity_mix(scored, "current"))
    print("Candidate severity mix (% of hands)")
    display(severity_mix(scored, "candidate"))



In [ ]:
if scored is None:
    print("Skipping composite swings: no re-score.")
else:
    scored = scored.copy()
    scored["composite_delta"] = scored["candidate_composite"] - scored["current_composite"]
    cols = ["source", "hand", "current_composite", "candidate_composite", "composite_delta"]
    display(
        scored.reindex(scored["composite_delta"].abs().sort_values(ascending=False).index)[cols].head(15)
    )



## 8. Hold-out gate (mandatory)

Compare candidate vs current on **hold-out** only:

- **Primary:** macro accuracy must **rise**
- **Tie-break / safety:** macro κ must not drop by more than `0.02`

`SAFE TO PROMOTE` only if both hold. Otherwise `DO NOT PROMOTE`.

The notebook still **does not write** `config/scoring.yaml`. A human copies the
snippet in the next section only after a safe verdict.



In [ ]:
KAPPA_SLACK = 0.02


def frame_with_prefix_severities(frame: pd.DataFrame, prefix: str) -> pd.DataFrame:
    out = frame.copy()
    for name in CRITERIA:
        out[f"severity_{name}"] = out[f"{prefix}_severity_{name}"]
        out[f"score_{name}"] = out[f"{prefix}_score_{name}"]
    out["composite_score"] = out[f"{prefix}_composite"]
    return out


def evaluate_from_frame(frame: pd.DataFrame) -> dict:
    with tempfile.TemporaryDirectory() as td:
        path = Path(td) / "summary.csv"
        frame.to_csv(path, index=False)
        return evaluate(path, LABELS_PATH, SPLIT_PATH, output_dir=None)


def _macro(report: dict, sl: str) -> tuple[float, float]:
    cell = (report.get("macro") or {}).get(sl) or {}
    return float(cell.get("accuracy", float("nan"))), float(cell.get("kappa", float("nan")))


if not HAS_EVAL:
    print("Skipping gate: technique_titan.eval is not importable.")
elif scored is None or split is None or baseline_report is None:
    print("Skipping gate: need batch summary, split, baseline, and a re-score.")
    if df is None:
        print(f"  {BATCH_CMD}")
else:
    candidate_report = evaluate_from_frame(frame_with_prefix_severities(scored, "candidate"))
    cur_acc, cur_k = _macro(baseline_report, "holdout")
    cand_acc, cand_k = _macro(candidate_report, "holdout")
    d_acc = cand_acc - cur_acc
    d_k = cand_k - cur_k
    acc_rises = np.isfinite(cand_acc) and np.isfinite(cur_acc) and cand_acc > cur_acc
    kappa_ok = np.isfinite(cand_k) and np.isfinite(cur_k) and cand_k >= (cur_k - KAPPA_SLACK)
    holdout_n = sum(
        int((candidate_report.get("criteria") or {}).get(name, {}).get("holdout", {}).get("n") or 0)
        for name in CRITERIA
    )

    if holdout_n == 0 or not np.isfinite(cand_acc) or not np.isfinite(cur_acc):
        PROMOTE_VERDICT = "DO NOT PROMOTE"
        GATE_REASON = "hold-out slice is empty or undefined"
    elif acc_rises and kappa_ok:
        PROMOTE_VERDICT = "SAFE TO PROMOTE"
        GATE_REASON = (
            f"hold-out macro accuracy rose ({cur_acc:.3f} → {cand_acc:.3f}) "
            f"and κ change {d_k:+.3f} is within slack {KAPPA_SLACK}"
        )
    elif not acc_rises:
        PROMOTE_VERDICT = "DO NOT PROMOTE"
        GATE_REASON = (
            f"hold-out macro accuracy did not rise ({cur_acc:.3f} → {cand_acc:.3f})"
        )
    else:
        PROMOTE_VERDICT = "DO NOT PROMOTE"
        GATE_REASON = (
            f"hold-out κ dropped materially ({cur_k:.3f} → {cand_k:.3f}, Δ={d_k:+.3f}; "
            f"slack={KAPPA_SLACK})"
        )

    print("HOLD-OUT GATE")
    print("-------------")
    print(f"current   macro acc={cur_acc:.3f}  κ={cur_k:.3f}")
    print(f"candidate macro acc={cand_acc:.3f}  κ={cand_k:.3f}")
    print(f"Δ acc={d_acc:+.3f}  Δ κ={d_k:+.3f}")
    print()
    print(f"Verdict: {PROMOTE_VERDICT}")
    print(f"Reason:  {GATE_REASON}")
    print(nfr_acc2_line(baseline_report, "current YAML"))
    print(nfr_acc2_line(candidate_report, "candidate (not shipped)"))
    print()
    print("Candidate per-criterion (train / holdout / all)")
    display(agreement_frame(candidate_report).round(3))
    print("Macro")
    display(macro_frame(candidate_report).round(3))

    side = agreement_frame(baseline_report).merge(
        agreement_frame(candidate_report),
        on=["criterion", "slice"],
        suffixes=("_current", "_candidate"),
    )
    print("Current vs candidate")
    display(side.round(3))
    print("Candidate hold-out confusion")
    show_confusion(candidate_report, "holdout")



## 9. Score curve for one metric

Pick a criterion to see how raw metric values map to 0–100 under current vs candidate.



In [ ]:
CRITERION = "wrist_height"  # change as needed

if df is None:
    print("Skipping score curve: no batch summary.")
    print(f"  {BATCH_CMD}")
else:
    cur_cfg = config["criteria"][CRITERION]
    cand_cfg = CANDIDATE["criteria"][CRITERION]
    metric = cur_cfg["metric"]
    values = df[metric].dropna()
    lo = min(values.min(), cur_cfg["limit"][0], cand_cfg["limit"][0])
    hi = max(values.max(), cur_cfg["limit"][1], cand_cfg["limit"][1])
    grid = pd.Series([lo + (hi - lo) * i / 200 for i in range(201)])

    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.plot(
        grid,
        [score_metric(v, cur_cfg["ideal"], cur_cfg["limit"]) for v in grid],
        label="current",
        color="#2c3e50",
    )
    ax.plot(
        grid,
        [score_metric(v, cand_cfg["ideal"], cand_cfg["limit"]) for v in grid],
        label="candidate",
        color="#c0392b",
        ls="--",
    )
    ax.scatter(
        values,
        [score_metric(v, cur_cfg["ideal"], cur_cfg["limit"]) for v in values],
        s=18,
        alpha=0.45,
        color="#4a6fa5",
        label="batch (current score)",
    )
    ax.axhline(bands["good_min"], color="#4caf50", lw=1, alpha=0.7)
    ax.axhline(bands["warning_min"], color="#e67e22", lw=1, alpha=0.7)
    ax.set_xlabel(metric)
    ax.set_ylabel("score")
    ax.set_title(f"Score mapping — {CRITERION}")
    ax.set_ylim(-5, 105)
    ax.legend()
    fig.tight_layout()
    plt.show()



## 10. Export candidate YAML snippet (copy by hand)

Prints candidate `criteria` + `severity_bands`. **Do not overwrite
`config/scoring.yaml` from this notebook** — paste only after `SAFE TO PROMOTE`.

Optionally writes `data/eval/reports/threshold_search.md` when the gate has run.
That file is a lab note, not an automatic config change.



In [ ]:
def _py(obj):
    if isinstance(obj, dict):
        return {k: _py(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_py(v) for v in obj]
    if isinstance(obj, (np.floating, np.integer)):
        return float(obj) if isinstance(obj, np.floating) else int(obj)
    if isinstance(obj, float):
        return float(obj)
    return obj


export = {
    "severity_bands": _py(CANDIDATE["severity_bands"]),
    "criteria": _py(CANDIDATE["criteria"]),
}
print(f"# Verdict: {PROMOTE_VERDICT}")
print("# Copy into config/scoring.yaml ONLY if the hold-out gate printed SAFE TO PROMOTE.")
print("# This notebook never writes that file.")
print(yaml.safe_dump(export, sort_keys=False))

if FIT_DETAILS is not None and not FIT_DETAILS.empty:
    print("Candidate metric table (train fit)")
    display(FIT_DETAILS)


def _md_table(frame: pd.DataFrame) -> str:
    cols = list(frame.columns)
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join("---" for _ in cols) + " |"]
    for _, row in frame.iterrows():
        cells = []
        for c in cols:
            v = row[c]
            if isinstance(v, float):
                cells.append(f"{v:.3f}" if np.isfinite(v) else "")
            else:
                cells.append(str(v))
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)


if baseline_report is not None and candidate_report is not None:
    REPORTS_DIR.mkdir(parents=True, exist_ok=True)
    report_path = REPORTS_DIR / "threshold_search.md"
    cur_acc, cur_k = _macro(baseline_report, "holdout")
    cand_acc, cand_k = _macro(candidate_report, "holdout")
    body = [
        "# Threshold search report",
        "",
        "Generated by `notebooks/scoring_tuning.ipynb`. Heuristics remain the scorer;",
        "this is calibration search, not a learned model.",
        "",
        f"**Verdict:** `{PROMOTE_VERDICT}`",
        "",
        GATE_REASON,
        "",
        "## Hold-out macro",
        "",
        f"- current accuracy={cur_acc:.3f} κ={cur_k:.3f}",
        f"- candidate accuracy={cand_acc:.3f} κ={cand_k:.3f}",
        "",
        nfr_acc2_line(baseline_report, "current YAML"),
        nfr_acc2_line(candidate_report, "candidate (not shipped)"),
        "",
        "## Baseline per-criterion",
        "",
        _md_table(agreement_frame(baseline_report).round(3)),
        "",
        "## Candidate per-criterion",
        "",
        _md_table(agreement_frame(candidate_report).round(3)),
        "",
        "## Candidate YAML (copy by hand only if SAFE TO PROMOTE)",
        "",
        "```yaml",
        yaml.safe_dump(export, sort_keys=False).rstrip(),
        "```",
        "",
    ]
    report_path.write_text("\n".join(body) + "\n")
    print(f"wrote {report_path}")
else:
    print("No gate report to write (baseline/candidate reports missing).")

